# Weather Data

# TODO: uv add for used packages

In [10]:
import cdsapi # important in order to use the CDS API
import pandas as pd
import numpy as np
import zipfile
import os
from glob import glob
import holidays

## CDS API Call 

In [11]:
# automatically read the last date from the taxi data set as end date for the weather data

taxi_data_processed = pd.read_parquet('../data/processed/taxi_data_processed.parquet')

end_date = pd.to_datetime(
    taxi_data_processed["Trip End Timestamp"].max(),
    format="%m/%d/%Y %I:%M:%S %p"
).strftime("%Y-%m-%d")

print(end_date)

2025-12-31


In [17]:
# config dictionary for the CDS API call

CONFIG = {
    # select the variables we are interested in; find the names of the variables at https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview or make use of the web based dataset picker 
    # on https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download 
    "variables": [
        "2m_temperature", 
        "total_precipitation",
        "snow_cover",
        "snow_depth",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "date": f"2024-01-01/{end_date}",
    "dir_name": "era5_data.zip",
}

In [19]:
# API call of the CDS API which can also be generated on https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download

dataset = "reanalysis-era5-land-timeseries"

# use the values from the config dictionary 
request = {
    "variable": CONFIG["variables"],
    "location": {"longitude": -87.8, "latitude": 42}, # coordinates of Chicago
    "date": CONFIG["date"],
    "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download(f"../data/{CONFIG['dir_name']}")

2026-07-03 15:14:32,444 INFO Request ID is 138b4361-9502-4aa5-a776-9b76c753ca1f
2026-07-03 15:14:32,529 INFO status has been updated to accepted
2026-07-03 15:14:46,274 INFO status has been updated to successful


25c32e03342aa72073bdc0b244d2d9e4.zip:   0%|          | 0.00/547k [00:00<?, ?B/s]

'../data/era5_data.zip'

## Load individual CSV files 
The data from the CDS API call is stored as collection of .CSV files in a ZIP file. Each weather variable is stored in its own .CSV file. The only expection are the wind variables. Both win components are stored in a single file.

In [21]:
# print what files are present in the zip
with zipfile.ZipFile(f"../data/{CONFIG['dir_name']}", "r") as zip_ref:
    zip_ref.extractall("era5_data")

print(os.listdir("era5_data"))

['reanalysis-era5-land-timeseries-sfc-2m-temperaturewg_vcvua.csv', 'reanalysis-era5-land-timeseries-sfc-pressure-precipitation_0bsw2ed.csv', 'reanalysis-era5-land-timeseries-sfc-snowk3cqj7x8.csv', 'reanalysis-era5-land-timeseries-sfc-wind6d6g2p7w.csv']


In [22]:
# list all CSV files
files = glob("era5_data/*.csv")

# dictionary for the dataframes
dfs = {}

for file in files:
    filename = os.path.basename(file)

    # find the correct csv based on the file names
    if "temperature" in filename:
        key = "temp"
    elif "precipitation" in filename:
        key = "precip"
    elif "snow" in filename:
        key = "snow"
    elif "wind" in filename:
        key = "wind"

    # load individual csv files 
    dfs[key] = pd.read_csv(file, encoding="latin1")

    print(f"\n--- {key} ---")
    print(dfs[key].head())


--- temp ---
            valid_time        t2m  latitude  longitude
0  2024-01-01 00:00:00  274.39615      42.0      -87.8
1  2024-01-01 01:00:00  274.24170      42.0      -87.8
2  2024-01-01 02:00:00  274.05762      42.0      -87.8
3  2024-01-01 03:00:00  273.90906      42.0      -87.8
4  2024-01-01 04:00:00  274.01672      42.0      -87.8

--- precip ---
            valid_time        tp  latitude  longitude
0  2024-01-01 00:00:00  0.000281      42.0      -87.8
1  2024-01-01 01:00:00  0.000150      42.0      -87.8
2  2024-01-01 02:00:00  0.000030      42.0      -87.8
3  2024-01-01 03:00:00  0.000014      42.0      -87.8
4  2024-01-01 04:00:00  0.000037      42.0      -87.8

--- snow ---
            valid_time     snowc       sde  latitude  longitude
0  2024-01-01 00:00:00  6.929688  0.007812      42.0      -87.8
1  2024-01-01 01:00:00  8.179688  0.008789      42.0      -87.8
2  2024-01-01 02:00:00  8.804688  0.008789      42.0      -87.8
3  2024-01-01 03:00:00  8.873047  0.008789    

## Merge individual Dataframes

In [23]:
df_merged = dfs["temp"].copy()

df_merged = df_merged.merge(
    dfs["precip"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["snow"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["wind"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)

df_merged

,valid_time,t2m,latitude,longitude,tp,snowc,sde,u10,v10
0,2024-01-01 00:00:00,274.39615,42.0,-87.8,0.000281,6.929688,0.007812,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,42.0,-87.8,0.000150,8.179688,0.008789,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,42.0,-87.8,0.000030,8.804688,0.008789,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,42.0,-87.8,0.000014,8.873047,0.008789,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,42.0,-87.8,0.000037,8.890625,0.008789,2.084305,-6.889282
...,...,...,...,...,...,...,...,...,...
17539,2025-12-31 19:00:00,275.06760,42.0,-87.8,0.000046,20.257812,0.020508,6.192612,1.700745
17540,2025-12-31 20:00:00,275.09650,42.0,-87.8,0.000083,20.263672,0.020508,6.181183,1.992905
17541,2025-12-31 21:00:00,275.02234,42.0,-87.8,0.000160,20.287110,0.020508,6.444611,1.568558
17542,2025-12-31 22:00:00,274.83923,42.0,-87.8,0.000214,20.525390,0.020508,6.504684,0.508759


## Data Preprocessing

In [24]:
# check for missing values
print("--- Missing values ---")
print(df_merged.isna().sum())
print("")

# check for data types
print("--- Data types ---")
print(df_merged.dtypes)
print("")

# short statistical description of the data
print("--- Stat description ---")
print(df_merged.describe())

--- Missing values ---
valid_time    0
t2m           0
latitude      0
longitude     0
tp            0
snowc         0
sde           0
u10           0
v10           0
dtype: int64

--- Data types ---
valid_time     object
t2m           float64
latitude      float64
longitude     float64
tp            float64
snowc         float64
sde           float64
u10           float64
v10           float64
dtype: object

--- Stat description ---
                t2m      latitude     longitude            tp         snowc  \
count  17544.000000  1.754400e+04  1.754400e+04  1.754400e+04  17544.000000   
mean     284.329026  4.200000e+01 -8.780000e+01  1.108098e-04      9.265091   
std       10.663718  1.129795e-12  2.408809e-11  5.460768e-04     24.963964   
min      248.497120  4.200000e+01 -8.780000e+01 -3.736932e-08      0.000000   
25%      275.900532  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
50%      285.043730  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
75%      

### Clean up

In [25]:
# drop unnecessary columns as always the same coordinates for all data points
df_merged = df_merged.drop(['latitude', 'longitude'], axis=1)

# rename columns to improve readability and interpretabilty
df_merged = df_merged.rename(
    columns={
        'valid_time': 'time_step',
        't2m': '2m_temp',
        'tp': 'total_precip',
        'snowc': 'snow_cov',
        'sde': 'snow_depth'
    }
)
df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282
...,...,...,...,...,...,...,...
17539,2025-12-31 19:00:00,275.06760,0.000046,20.257812,0.020508,6.192612,1.700745
17540,2025-12-31 20:00:00,275.09650,0.000083,20.263672,0.020508,6.181183,1.992905
17541,2025-12-31 21:00:00,275.02234,0.000160,20.287110,0.020508,6.444611,1.568558
17542,2025-12-31 22:00:00,274.83923,0.000214,20.525390,0.020508,6.504684,0.508759


In [26]:
# change dtype to datetime object for column time_step
df_merged['time_step'] = pd.to_datetime(df_merged['time_step'])

# set lower boundary for precipitation and snow depth to get rid of physically impossible values, e.g. negative preciptation (porbably due to accumulation of small floating point operation errors)
df_merged['total_precip'] = df_merged['total_precip'].clip(lower=0)
df_merged['snow_depth'] = df_merged['snow_depth'].clip(lower=0)

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282
...,...,...,...,...,...,...,...
17539,2025-12-31 19:00:00,275.06760,0.000046,20.257812,0.020508,6.192612,1.700745
17540,2025-12-31 20:00:00,275.09650,0.000083,20.263672,0.020508,6.181183,1.992905
17541,2025-12-31 21:00:00,275.02234,0.000160,20.287110,0.020508,6.444611,1.568558
17542,2025-12-31 22:00:00,274.83923,0.000214,20.525390,0.020508,6.504684,0.508759


### Conversion of units

In [27]:
# convert temperature from Kelvin to Celsius via formula C = K - 273.15
df_merged['2m_temp_c'] = df_merged['2m_temp'] - 273.15

# convert from m to mm (common unit for precipitation)
df_merged['total_precip_mm'] = df_merged['total_precip'] * 1000

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491
...,...,...,...,...,...,...,...,...,...
17539,2025-12-31 19:00:00,275.06760,0.000046,20.257812,0.020508,6.192612,1.700745,1.91760,0.045590
17540,2025-12-31 20:00:00,275.09650,0.000083,20.263672,0.020508,6.181183,1.992905,1.94650,0.083260
17541,2025-12-31 21:00:00,275.02234,0.000160,20.287110,0.020508,6.444611,1.568558,1.87234,0.160234
17542,2025-12-31 22:00:00,274.83923,0.000214,20.525390,0.020508,6.504684,0.508759,1.68923,0.213832


## Data Export
Export the data into its own CSV file in order to merge it with POI and taxi data later on. 

In [29]:
# Export weather data table to use in other notebooks
df_merged.to_parquet("../data/processed/weather_data_processed.parquet")